In [ ]:
# ==========================================
# 1. INSTALLATIONS & IMPORTS
# ==========================================
!pip install -q transformers datasets torch scikit-learn accelerate sentencepiece protobuf

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import zipfile
import os
import shutil
from dataclasses import dataclass
from typing import Optional, Tuple, Dict, List, Any
from collections import Counter, defaultdict
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    DebertaV2PreTrainedModel,
    DebertaV2Model,
    Trainer,
    TrainingArguments,
    EvalPrediction,
    EarlyStoppingCallback  # <--- Added for Early Stopping
)
from transformers.modeling_outputs import SequenceClassifierOutput
from datasets import load_dataset, load_from_disk, DatasetDict, concatenate_datasets

# Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==========================================
# 2. CONFIGURATION
# ==========================================
MODEL_ID = "microsoft/deberta-v3-base"

# Model Parameters
MAX_LENGTH = 256
CHUNK_STRIDE = 160
MAX_CHUNKS = 16

# Memory Optimization
BATCH_SIZE = 16
GRAD_ACCUMULATION = 1
GRADIENT_CHECKPOINTING = True

# Training Hyperparameters
LEARNING_RATE = 1e-5
WARMUP_RATIO = 0.1
FOCAL_LOSS_GAMMA = 2.0

# ==========================================
# 3. DATA LOADING (OVERSAMPLING REMOVED)
# ==========================================
print("\n--- Processing Data ---")

# 1. Unzip
zip_path = "processed_dataset.zip"
extract_path = "./processed_dataset_extracted"

if not os.path.exists(extract_path):
    print(f"Unzipping {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

# 2. Load from Disk
try:
    dataset = load_from_disk(extract_path)
except:
    potential_path = os.path.join(extract_path, "processed_dataset")
    if os.path.exists(potential_path):
        dataset = load_from_disk(potential_path)
    else:
        if os.path.exists(os.path.join(extract_path, "train")):
            dataset = load_from_disk(extract_path)
        else:
            raise FileNotFoundError("Could not locate the dataset inside the extracted folder.")

print(f"Dataset loaded: {dataset}")

# 3. Filter Valid Rows
train_df = dataset['train'].filter(lambda x: x['evasion_label'] is not None and x['evasion_label'] != "")
labels_list = sorted(set(train_df['evasion_label']))
num_labels = len(labels_list)
label2id = {l: i for i, l in enumerate(labels_list)}
id2label = {i: l for i, l in enumerate(labels_list)}

print(f"Labels map: {label2id}")

def encode_train_labels(batch):
    return {"labels": label2id[batch['evasion_label']]}

train_dataset = train_df.map(encode_train_labels)

# 4. Calculate Class Weights (Based on Original Distribution)
# Since oversampling is removed, we calculate weights on the raw train_dataset
train_labels_list = train_dataset['labels']
label_counts = Counter(train_labels_list)
total_samples = len(train_labels_list)

class_weights = []
for i in range(num_labels):
    count = label_counts.get(i, 0)
    if count == 0: count = 1
    weight = total_samples / (num_labels * count)
    class_weights.append(weight)

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f"Class Weights: {class_weights}")

# ==========================================
# 4. MIL DATASET
# ==========================================
class MILChunkingDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_len, stride, max_chunks, is_test=False):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.stride = stride
        self.max_chunks = max_chunks
        self.is_test = is_test

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        row = self.dataset[idx]

        q_text = row['formatted_question']
        long_answer = row['interview_answer']

        if self.is_test:
            label = 0
        else:
            label = row['labels']

        # Tokenize answer
        answer_tokens = self.tokenizer(long_answer, add_special_tokens=False)['input_ids']
        tokens_per_chunk = self.max_len - 120

        if len(answer_tokens) == 0:
            windows = [[]]
        else:
            windows = [
                answer_tokens[i : i + tokens_per_chunk]
                for i in range(0, len(answer_tokens), self.stride)
            ]
            windows = windows[:self.max_chunks]

        chunk_input_ids = []
        chunk_attention_masks = []
        chunk_token_type_ids = []

        for window in windows:
            window_text = self.tokenizer.decode(window)

            # Contextual Markers
            formatted_input_text = f"QUESTION: {q_text} [SEP] ANSWER: {window_text}"

            encoded = self.tokenizer(
                formatted_input_text,
                padding='max_length',
                truncation=True,
                max_length=self.max_len,
                return_tensors='pt'
            )

            chunk_input_ids.append(encoded['input_ids'].squeeze(0))
            chunk_attention_masks.append(encoded['attention_mask'].squeeze(0))

            if 'token_type_ids' in encoded:
                chunk_token_type_ids.append(encoded['token_type_ids'].squeeze(0))
            else:
                chunk_token_type_ids.append(torch.zeros_like(encoded['input_ids'].squeeze(0)))

        return {
            "input_ids": torch.stack(chunk_input_ids),
            "attention_mask": torch.stack(chunk_attention_masks),
            "token_type_ids": torch.stack(chunk_token_type_ids),
            "labels": torch.tensor(label, dtype=torch.long)
        }

# ==========================================
# 5. DATA COLLATOR
# ==========================================
@dataclass
class MILDataCollator:
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        max_chunks_batch = max(f["input_ids"].shape[0] for f in features)

        batch_input_ids = []
        batch_masks = []
        batch_token_types = []
        batch_labels = []

        for f in features:
            source_ids = f["input_ids"]
            source_mask = f["attention_mask"]
            source_types = f["token_type_ids"]
            num_chunks, seq_len = source_ids.shape

            pad_chunks = max_chunks_batch - num_chunks
            if pad_chunks > 0:
                pad_ids = torch.zeros((pad_chunks, seq_len), dtype=torch.long)
                pad_mask = torch.zeros((pad_chunks, seq_len), dtype=torch.long)
                pad_types = torch.zeros((pad_chunks, seq_len), dtype=torch.long)

                padded_ids = torch.cat([source_ids, pad_ids], dim=0)
                padded_mask = torch.cat([source_mask, pad_mask], dim=0)
                padded_types = torch.cat([source_types, pad_types], dim=0)
            else:
                padded_ids = source_ids
                padded_mask = source_mask
                padded_types = source_types

            batch_input_ids.append(padded_ids)
            batch_masks.append(padded_mask)
            batch_token_types.append(padded_types)
            batch_labels.append(f["labels"])

        return {
            "input_ids": torch.stack(batch_input_ids),
            "attention_mask": torch.stack(batch_masks),
            "token_type_ids": torch.stack(batch_token_types),
            "labels": torch.stack(batch_labels)
        }

# ==========================================
# 6. MODEL (FIXED CHECKPOINTING)
# ==========================================
class DebertaForMIL(DebertaV2PreTrainedModel):
    def __init__(self, config, class_weights=None, gamma=2.0):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.gamma = gamma
        self.class_weights = class_weights

        self.deberta = DebertaV2Model(config)
        self.attention_layer = nn.Linear(config.hidden_size, 1)

        drop_prob = getattr(config, "cls_dropout", None)
        drop_prob = drop_prob if drop_prob is not None else config.hidden_dropout_prob
        self.dropout = nn.Dropout(drop_prob)

        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

        if getattr(config, "gradient_checkpointing", False):
            self.deberta.gradient_checkpointing_enable()

        self.post_init()

    def gradient_checkpointing_enable(self, gradient_checkpointing_kwargs=None):
        self.deberta.gradient_checkpointing_enable(gradient_checkpointing_kwargs=gradient_checkpointing_kwargs)

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, labels=None, **kwargs):
        batch_size, num_chunks, seq_len = input_ids.shape
        flat_input_ids = input_ids.view(-1, seq_len)
        flat_mask = attention_mask.view(-1, seq_len)
        flat_token_types = token_type_ids.view(-1, seq_len) if token_type_ids is not None else None

        outputs = self.deberta(
            input_ids=flat_input_ids,
            attention_mask=flat_mask,
            token_type_ids=flat_token_types
        )

        cls_output = outputs.last_hidden_state[:, 0, :]
        attn_scores = self.attention_layer(cls_output).view(batch_size, num_chunks)
        chunk_mask = torch.any(attention_mask > 0, dim=-1)

        attn_scores = attn_scores.masked_fill(~chunk_mask, -65000.0)
        attn_weights = F.softmax(attn_scores, dim=1)

        cls_output_reshaped = cls_output.view(batch_size, num_chunks, -1)
        context_vector = torch.sum(cls_output_reshaped * attn_weights.unsqueeze(-1), dim=1)

        logits = self.classifier(self.dropout(context_vector))

        loss = None
        if labels is not None:
            if self.class_weights is not None:
                self.class_weights = self.class_weights.to(logits.device)

            ce_loss = F.cross_entropy(logits, labels, weight=self.class_weights, reduction='none')
            pt = torch.exp(-ce_loss)
            focal_loss = ((1 - pt) ** self.gamma) * ce_loss
            loss = focal_loss.mean()

        return SequenceClassifierOutput(loss=loss, logits=logits)

# ==========================================
# 7. CUSTOM TRAINER
# ==========================================
class MultiAnnotatorTrainer(Trainer):
    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval"):
        eval_dataset = eval_dataset if eval_dataset is not None else self.eval_dataset
        output = self.predict(eval_dataset, metric_key_prefix="test")

        preds = np.argmax(output.predictions, axis=1)
        pred_labels = [self.model.config.id2label[p] for p in preds]
        hf_test_data = eval_dataset.dataset

        tp = defaultdict(int)
        fp = defaultdict(int)
        fn = defaultdict(int)
        all_classes = set()

        for i, pred_label in enumerate(pred_labels):
            anns = [
                hf_test_data[i]['annotator1'],
                hf_test_data[i]['annotator2'],
                hf_test_data[i]['annotator3']
            ]
            true_set = set([a for a in anns if a])
            all_classes.add(pred_label)
            all_classes.update(true_set)

            if pred_label in true_set:
                tp[pred_label] += 1
            else:
                fp[pred_label] += 1
                for true_cls in true_set:
                    fn[true_cls] += 1

        f1_scores = []
        for cls in all_classes:
            class_tp = tp[cls]
            class_fp = fp[cls]
            class_fn = fn[cls]
            prec = class_tp / (class_tp + class_fp) if (class_tp + class_fp) > 0 else 0.0
            rec = class_tp / (class_tp + class_fn) if (class_tp + class_fn) > 0 else 0.0
            f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
            f1_scores.append(f1)

        macro_f1 = sum(f1_scores) / len(f1_scores) if f1_scores else 0.0

        metrics = {f"{metric_key_prefix}_f1_macro": macro_f1}
        self.log(metrics)
        self.control = self.callback_handler.on_evaluate(self.args, self.state, self.control, metrics)
        return metrics

# ==========================================
# 8. EXECUTION
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Prepare Datasets (Using Original Train Dataset, NO Oversampling)
train_ds = MILChunkingDataset(train_dataset, tokenizer, MAX_LENGTH, CHUNK_STRIDE, MAX_CHUNKS, is_test=False)
test_ds = MILChunkingDataset(dataset['test'], tokenizer, MAX_LENGTH, CHUNK_STRIDE, MAX_CHUNKS, is_test=True)

# Initialize Model
model = DebertaForMIL.from_pretrained(
    MODEL_ID,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    class_weights=class_weights_tensor,
    gamma=FOCAL_LOSS_GAMMA
)

# Training Arguments
training_args = TrainingArguments(
    output_dir="./Deberta_Large_MIL_NoOversample_EarlyStopping",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,

    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    num_train_epochs=15,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_macro",
    greater_is_better=True,
    logging_steps=25,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = MultiAnnotatorTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=MILDataCollator(),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)] # <--- Added Patience
)

print("\n--- Starting Training (DeBERTa + MIL + Early Stopping + No Oversampling) ---")
trainer.train()

# ==========================================
# 9. FINAL REPORT
# ==========================================
print("\n" + "="*50)
print("FINAL EVALUATION")
print("="*50)

output = trainer.predict(test_ds)
preds = np.argmax(output.predictions, axis=1)
pred_labels = [id2label[p] for p in preds]

hf_test_data = dataset['test']
tp = defaultdict(int)
fp = defaultdict(int)
fn = defaultdict(int)
all_classes = set()

for i, pred_label in enumerate(pred_labels):
    anns = [
        hf_test_data[i]['annotator1'],
        hf_test_data[i]['annotator2'],
        hf_test_data[i]['annotator3']
    ]
    true_set = set([a for a in anns if a])
    all_classes.add(pred_label)
    all_classes.update(true_set)
    if pred_label in true_set:
        tp[pred_label] += 1
    else:
        fp[pred_label] += 1
        for true_cls in true_set:
            fn[true_cls] += 1

f1_scores = []
all_classes_sorted = sorted(list(all_classes))

col_1_width = max(max(len(str(cls)) for cls in all_classes_sorted), len("Class")) + 2
header = (
    f"{'Class':<{col_1_width}} | {'TP':<5} | {'FP':<5} | {'FN':<5} | "
    f"{'Precision':<10} | {'Recall':<10} | {'F1-Score':<10}"
)

print(header)
print("-" * len(header))

for cls in all_classes_sorted:
    class_tp = tp[cls]
    class_fp = fp[cls]
    class_fn = fn[cls]
    prec = class_tp / (class_tp + class_fp) if (class_tp + class_fp) > 0 else 0.0
    rec = class_tp / (class_tp + class_fn) if (class_tp + class_fn) > 0 else 0.0
    f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
    f1_scores.append(f1)
    print(
        f"{cls:<{col_1_width}} | {class_tp:<5} | {class_fp:<5} | {class_fn:<5} | "
        f"{prec:<10.4f} | {rec:<10.4f} | {f1:<10.4f}"
    )

macro_f1 = sum(f1_scores) / len(f1_scores) if f1_scores else 0.0
print("-" * len(header))
print(f"FINAL MACRO F1 SCORE: {macro_f1:.6f}")
print("="*len(header))

# Save
trainer.save_model("./Final_Deberta_MIL_NoOversample")
tokenizer.save_pretrained("./Final_Deberta_MIL_NoOversample")

# Generate Predictions
output_file = "prediction"
with open(output_file, "w") as f:
    for label in pred_labels:
        f.write(f"{label}\n")

print(f"✅ Saved predictions to '{output_file}'")
!head -n 5 prediction

In [ ]:
# ==========================================
# 10. RUN MODEL ON NEW CSV (clarity_task_evaluation_dataset.csv)
# ==========================================
import pandas as pd
from datasets import Dataset

# Load trained model
model = DebertaForMIL.from_pretrained("./Final_Deberta_MIL_NoOversample").to(device)
tokenizer = AutoTokenizer.from_pretrained("./Final_Deberta_MIL_NoOversample")
model.eval()

# Load new CSV
df = pd.read_csv("clarity_task_evaluation_dataset.csv")

# Convert to HF Dataset
hf_eval = Dataset.from_pandas(df)

# Rename column: question -> formatted_question
hf_eval = hf_eval.rename_column("question", "formatted_question")

# Build MIL Dataset (no labels needed)
eval_ds = MILChunkingDataset(hf_eval, tokenizer, MAX_LENGTH, CHUNK_STRIDE, MAX_CHUNKS, is_test=True)

# Run inference
trainer = Trainer(model=model, data_collator=MILDataCollator())
output = trainer.predict(eval_ds)

# Decode predictions
preds = np.argmax(output.predictions, axis=1)
pred_labels = [id2label[p] for p in preds]

# Save extensionless file "prediction"
with open("prediction", "w") as f:
    for label in pred_labels:
        f.write(f"{label}\n")

print("✅ Saved predictions to 'prediction'")
!head -n 5 prediction